# 🚀 ComfyUI + MiniMax-H3 on Google Colab (A100 GPU Edition)

Google AI Pro 等のプランで付与される **Colab Compute Units (CU)** を活用して、強力な最新動画生成モデル **MiniMax-H3 (Hailuo)** を A100 (40GB VRAM) 環境で動かすためのオールインワン検証テンプレートです。

> 📖 **詳細な解説・検証記事 (Zenn)**: [Google AI Pro(2,900円)に課金するとA100 40GBがColab経由で毎月37時間分使える話](https://zenn.dev/grand2/articles/ddba80ba400f6f)

### 💡 特徴・ポイント
- **A100 GPU 最適化**: MiniMax-H3 の大規模モデル (`int8_convrot` 等) + Text Encoder + Audio/Video VAE をストレスなくロード。
- **🚀 Turbo LoRA 最適化 (推奨)**: 通常 25 ステップ（約30分）かかる生成を、**わずか 6〜8 ステップ（約7分・所要時間 1/4）へ激変** させる Turbo LoRA を自動配備。
- **高速化スタック対応**: `SageAttention`、`TeaCache` などの最新高速化ノード群をトグル1つで自動セットアップ。
- **Google Drive 永続化 & ローカル SSD 最適化**:
  - モデルは Colab のローカル SSD 上に配置し、Drive (FUSE) 経由の読み込みによる WebUI の低速化・フリーズを回避。
  - モデル・LoRA（約20GB）は `MyDrive/ComfyUI_Models/` にキャッシュし、2回目以降は HF からの DL をスキップして Drive → ローカル SSD へコピー。
  - 生成された動画・画像は **`MyDrive/ComfyUI_Outputs/` に自動保存**。ランタイムが切断・終了しても成果物が消えません。
- **Cloudflare Tunnel (無料・トークン不要)**: ngrok 不要ですぐにセキュアな一時公開 URL (`trycloudflare.com`) を自動発行。

> ⚠️ **注意**: ノートブック上部のメニュー「ランタイム」→「ランタイムのタイプを変更」から、**GPU (A100)** が選択されていることを確認してください。

## Step 1: 環境確認 & Google Drive マウント (永続化連携)

In [ ]:
# GPU および CUDA バージョンの確認 (A100 がアサインされているか確認)
!nvidia-smi

# @markdown **USE_GOOGLE_DRIVE**: ON にすると Google Drive をマウントし、以下を行います（OFF の場合はすべてランタイム内のみで完結し、切断時に消えます）。
# @markdown - モデルを `MyDrive/ComfyUI_Models/` にキャッシュ（2回目以降は DL 不要）
# @markdown - 生成物を `MyDrive/ComfyUI_Outputs/` に自動保存
# @markdown - `MyDrive/ComfyUI_Inputs/` の素材を起動時に読み込み
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")

## Step 2: ComfyUI 本体 & 高速化スタックのセットアップ

- **ENABLE_ACCELERATION (推奨)**: `SageAttention` / `Triton` をインストールします。
- **INSTALL_TEACACHE (推奨)**: `TeaCache` 高速化ノードをインストールします。
- **CUDA 12.8 / 13.0 互換性ハンドリング**: PyTorch cu128 とホスト CUDA 13.0 の互換レイヤーを安全に処理します。

In [ ]:
import os

# @title 高速化オプション設定
# @markdown **ENABLE_ACCELERATION**: SageAttention / Triton をインストールします（推奨）。
ENABLE_ACCELERATION = True  # @param {type:"boolean"}
# @markdown **INSTALL_TEACACHE**: TeaCache 高速化ノードをインストールします（推奨）。
INSTALL_TEACACHE = True     # @param {type:"boolean"}

%cd /content

# ComfyUI 本体のクローン (最新版)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# 依存ライブラリのインストール
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q huggingface_hub

# SageAttention & Triton セットアップ
if ENABLE_ACCELERATION:
    print("⚡ 高速化スタック (SageAttention, Triton 等) をセットアップ中...")
    !pip install -q triton sageattention

# ComfyUI-Manager のインストール
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# MiniMax-H3 Easy ノード
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

# TeaCache 高速化ノード
if INSTALL_TEACACHE and not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-TeaCache"):
    !git clone https://github.com/chengzeyi/ComfyUI-MiniMaxH3-TeaCache.git || true

%cd /content/ComfyUI
print("✅ ノードおよび依存ライブラリの準備が完了しました！")

## Step 3: MiniMax-H3 モデル & Turbo LoRA の準備 (ローカル SSD 配置 + Drive キャッシュ)

- **🚀 Turbo LoRA (`DOWNLOAD_TURBO_LORA = True`)**: `lightx2v/Minimax-h3-Turbo` をダウンロードし、**8ステップサンプリング（生成時間1/4）** を可能にします。
- **⚡ モデルはローカル SSD に配置**: Drive 上のモデルを直接読むと WebUI の起動・操作が大幅に遅くなるため、モデルの実体は常に `/content/ComfyUI/models/` に置きます。
- `USE_GOOGLE_DRIVE = True` の場合：
  - **モデルキャッシュ**: `MyDrive/ComfyUI_Models/` にキャッシュがあればローカルへ並列コピー（Drive は 1 ストリームだと遅いため分割して同時に読み込み）、なければ Hugging Face から DL して Drive にも保存します（次回以降 DL スキップ）。
  - **動画出力先**: `MyDrive/ComfyUI_Outputs/` (インスタンス終了後も成果物を保持)
  - **入力素材**: `MyDrive/ComfyUI_Inputs/` の素材を起動時にローカルへコピーします。**WebUI からアップロードした素材は Drive に保存されない** ので、残したい素材は Drive 側のフォルダに置いてください。

In [ ]:
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download

# @title Turbo LoRA ダウンロード設定
# @markdown **DOWNLOAD_TURBO_LORA**: 8ステップ生成用の Turbo LoRA をダウンロードします（推奨）。
DOWNLOAD_TURBO_LORA = True  # @param {type:"boolean"}

LOCAL_MODELS_DIR = "/content/ComfyUI/models"
LOCAL_OUTPUT_DIR = "/content/ComfyUI/output"
LOCAL_INPUT_DIR = "/content/ComfyUI/input"

DRIVE_BASE = "/content/drive/MyDrive"
DRIVE_MODELS_DIR = os.path.join(DRIVE_BASE, "ComfyUI_Models")
DRIVE_OUTPUT_DIR = os.path.join(DRIVE_BASE, "ComfyUI_Outputs")
DRIVE_INPUT_DIR = os.path.join(DRIVE_BASE, "ComfyUI_Inputs")

MIN_MODEL_SIZE = 1_000_000
COPY_WORKERS = 8          # Drive からの並列読み込み数
COPY_BLOCK = 64 << 20     # 並列コピーの分割単位 (64MB)

def rsync(src, dst):
    !rsync -a -h --info=progress2 "{src}" "{dst}"
    if _exit_code != 0:
        raise RuntimeError(f"rsync に失敗しました (exit {_exit_code}): {src} -> {dst}")

def parallel_copy(src, dst, workers=COPY_WORKERS):
    # Drive (FUSE) は 1 ストリームあたりの読み込みが遅いため、ファイルをブロックに分割して並列に読む
    size = os.path.getsize(src)
    tmp = dst + ".part"
    blocks = [(off, min(COPY_BLOCK, size - off)) for off in range(0, size, COPY_BLOCK)]
    lock = threading.Lock()
    copied = 0
    start = time.time()

    with open(tmp, "wb") as f:
        f.truncate(size)

    def copy_block(block):
        nonlocal copied
        off, length = block
        src_fd = os.open(src, os.O_RDONLY)
        dst_fd = os.open(tmp, os.O_WRONLY)
        try:
            done = 0
            while done < length:
                buf = os.pread(src_fd, length - done, off + done)
                if not buf:
                    raise IOError(f"予期せぬ EOF: {src} (offset {off + done})")
                view = memoryview(buf)
                while view:
                    written = os.pwrite(dst_fd, view, off + done)
                    view = view[written:]
                    done += written
        finally:
            os.close(src_fd)
            os.close(dst_fd)
        with lock:
            copied += length
            elapsed = max(time.time() - start, 1e-6)
            print(f"\r  {copied / 1e9:.1f} / {size / 1e9:.1f} GB  ({copied / 1e6 / elapsed:.0f} MB/s)", end="", flush=True)

    try:
        with ThreadPoolExecutor(workers) as ex:
            list(ex.map(copy_block, blocks))
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise
    print()
    if os.path.getsize(tmp) != size:
        os.remove(tmp)
        raise RuntimeError(f"コピー後のサイズが一致しません: {src}")
    os.replace(tmp, dst)

def is_valid_model(path):
    return os.path.isfile(path) and os.path.getsize(path) > MIN_MODEL_SIZE

def ensure_local_dir(path):
    # 旧バージョンで作られた Drive へのシンボリックリンクを外し、ローカルの実ディレクトリにする
    if os.path.islink(path):
        os.unlink(path)
    os.makedirs(path, exist_ok=True)

# 1. 出力先: Google Drive にシンボリックリンク (生成物を即時永続化)
if USE_GOOGLE_DRIVE:
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    if os.path.exists(LOCAL_OUTPUT_DIR) and not os.path.islink(LOCAL_OUTPUT_DIR):
        !rm -rf {LOCAL_OUTPUT_DIR}
    if not os.path.exists(LOCAL_OUTPUT_DIR):
        os.symlink(DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR)
    print(f"📁 出力先を Google Drive に連携: {DRIVE_OUTPUT_DIR}")
else:
    ensure_local_dir(LOCAL_OUTPUT_DIR)

# 2. 入力素材: Google Drive からローカル SSD へコピー (リンクすると WebUI の一覧表示が遅くなるため)
ensure_local_dir(LOCAL_INPUT_DIR)
if USE_GOOGLE_DRIVE:
    os.makedirs(DRIVE_INPUT_DIR, exist_ok=True)
    print(f"🖼️ 入力素材を Google Drive からコピー中: {DRIVE_INPUT_DIR}")
    rsync(DRIVE_INPUT_DIR + "/", LOCAL_INPUT_DIR + "/")

# 3. モデル定義
REPO_ID = "Comfy-Org/MiniMax-H3"
files_to_download = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", REPO_ID),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", REPO_ID),
    ("vae/minimax_h3_video_vae_fp16.safetensors", REPO_ID),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", REPO_ID)
]

if DOWNLOAD_TURBO_LORA:
    files_to_download.append((
        "loras/minimax_h3_fl2v_turbo_8step_v1.0_768p_bf16.safetensors",
        "lightx2v/Minimax-h3-Turbo"
    ))

# 4. モデルをローカル SSD に配置 (ローカル → Drive キャッシュ → Hugging Face の順に探す)
print("📥 MiniMax-H3 モデル & Turbo LoRA の確認・配置を開始...")
for sub in sorted({os.path.dirname(f) for f, _ in files_to_download}):
    ensure_local_dir(os.path.join(LOCAL_MODELS_DIR, sub))

for filename, repo in files_to_download:
    local_path = os.path.join(LOCAL_MODELS_DIR, filename)
    drive_path = os.path.join(DRIVE_MODELS_DIR, filename)
    sub_folder = os.path.dirname(filename)
    file_name_only = os.path.basename(filename)

    if is_valid_model(local_path):
        print(f"⚡ ローカルSSDに配置済み: {filename}")
    elif USE_GOOGLE_DRIVE and is_valid_model(drive_path):
        print(f"📦 Drive キャッシュからローカルSSDへ並列コピー中 ({COPY_WORKERS}並列): {filename} ...")
        parallel_copy(drive_path, local_path)
    else:
        print(f"⬇️ Hugging Face からダウンロード中: {filename} (from {repo}) ...")
        hf_hub_download(
            repo_id=repo,
            filename=file_name_only if repo != REPO_ID else filename,
            local_dir=os.path.join(LOCAL_MODELS_DIR, sub_folder) if repo != REPO_ID else LOCAL_MODELS_DIR,
        )

    if not is_valid_model(local_path):
        raise RuntimeError(f"モデルの配置に失敗しました: {local_path}")

    # 次回起動の高速化のため Drive にキャッシュ (未キャッシュの場合のみ)
    if USE_GOOGLE_DRIVE and not is_valid_model(drive_path):
        os.makedirs(os.path.dirname(drive_path), exist_ok=True)
        print(f"💾 次回起動用に Google Drive へキャッシュ中: {drive_path} ...")
        rsync(local_path, drive_path)

print("✅ すべてのモデルおよび Turbo LoRA のローカル SSD 配置が完了しました！")

## Step 4: ComfyUI 起動 & Cloudflare Tunnel 経由でアクセス

バックグラウンドで ComfyUI を起動し、Cloudflare Tunnel (`trycloudflare.com`) の安全なパブリック URL を発行して**起動状態を維持**します。
表示された `https://xxxx.trycloudflare.com` のリンクをクリックすると ComfyUI の WebUI が開きます。

### 💡 Turbo LoRA を使った超高速ワークフローの組み方
1. ComfyUI を開いたら、モデルローダー（`UNETLoader`）の直後に **`LoraLoader`** を挟みます。
2. LoRA に **`minimax_h3_fl2v_turbo_8step_...safetensors`** を選択します。
3. **KSampler の steps を `25` → `8` に変更** します（Euler / normal 推奨）。
4. これで **約 6〜8 分で 1 本の美麗動画が完成** します！

In [ ]:
import subprocess
import threading
import time
import re

# Cloudflared のダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# ComfyUI をバックグラウンドで起動
print("⚡ Starting ComfyUI...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# ComfyUI の起動ログを別スレッドで監視
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# 少し待機して Cloudflared トンネルを起動
time.sleep(5)
print("🌐 Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not url_found:
        url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 ComfyUI is LIVE: {url}")
        print("="*60 + "\n")
        print("💡 サーバーを稼働維持しています。(終了したい場合は本セルの停止ボタンを押すか、上部メニューから切断してください)")
        url_found = True

# トンネルプロセスが生きている間はセルを終了させず維持
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ サーバーを停止しました。")
    comfy_proc.terminate()
    tunnel_proc.terminate()

## 🛑 (作業完了時) 確実に課金をストップする方法

動画生成の検証が完了したら、**最も確実に Compute Units (CU) の消費をストップ** するため、以下の手順でインスタンスを破棄してください：

### 💡 【推奨・確実度 100%】Colab メニューから切断
1. Colab 画面上部メニューの **「ランタイム」** をクリック
2. **「ランタイムの接続を解除して削除」** を選択（確認画面で「はい」）

> `USE_GOOGLE_DRIVE = True` の場合、成果物（動画）は Google Drive（`MyDrive/ComfyUI_Outputs/`）にリアルタイム保存されているため、切断しても安全です。
> `False` にしている場合、または WebUI からアップロードした入力素材はランタイム削除とともに消えるので、必要なものは事前にダウンロードしてください。